In [2]:
import pandas as pd
import numpy as np

In [3]:
movies_df = pd.read_csv('/Users/ahmedsherif/Downloads/ml-25m/movies.csv')
ratings_df = pd.read_csv('/Users/ahmedsherif/Downloads/ml-25m/ratings.csv')

In [4]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [5]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
min_movies_rating = 50
min_user_rating = 20

active_users = ratings_df['userId'].value_counts()
active_users = active_users[active_users >= min_user_rating].index

active_movies = ratings_df['movieId'].value_counts()
active_movies = active_movies[active_movies >= min_movies_rating].index

filtered_ratings = ratings_df[ratings_df['userId'].isin(active_users) & ratings_df['movieId'].isin(active_movies)]

In [7]:
merged_df = pd.merge(movies_df,filtered_ratings,on = 'movieId')
merged_df.head()

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,2,3.5,1141415820
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3,4.0,1439472215
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,4,3.0,1573944252
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0,858625949
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,8,4.0,890492517


In [8]:
n_users = merged_df['userId'].nunique()
n_items = merged_df['movieId'].nunique()

In [9]:
total_cells = n_users * n_items 

In [10]:
n_ratings = len(merged_df)

In [11]:
sparsity = ((1 - n_ratings / total_cells)*100)
sparsity

98.8492427701137

In [12]:
from surprise import Reader , Dataset , SVD
from surprise.model_selection import cross_validate

sample_df = merged_df.sample(100000,random_state=42)

reader = Reader(rating_scale = (0.5 , 5))

data = Dataset.load_from_df(sample_df[['userId' , 'movieId' , 'rating']],reader)

algo = SVD(n_factors = 50 , reg_all = 0.02, lr_all = 0.1 , n_epochs = 20 , random_state = 20)

results = cross_validate(algo , data , measures=['MSE' , 'RMSE'],cv = 5 , verbose=True)

Evaluating MSE, RMSE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
MSE (testset)     0.9698  0.9443  0.9472  0.9568  0.9608  0.9558  0.0092  
RMSE (testset)    0.9848  0.9718  0.9732  0.9782  0.9802  0.9776  0.0047  
Fit time          0.34    0.35    0.37    0.39    0.39    0.37    0.02    
Test time         0.04    0.07    0.04    0.08    0.04    0.05    0.02    


In [13]:
trainset = data.build_full_trainset()
prediction = algo.predict(uid = 1 , iid = 289)
print(prediction.est)

3.398390965642918


In [14]:
def recommend(user_id , n = 10):
    rated_movies = set(filtered_ratings[filtered_ratings['userId'] == user_id]['movieId'])
    all_movies = set(merged_df['movieId'].unique())
    unseen_movies = all_movies - rated_movies 

    predictions = []
    for movie in unseen_movies:
        pred = algo.predict(uid = user_id , iid = movie)
        predictions.append((movie , pred.est))
    predictions.sort(key = lambda x : x[1] , reverse = True)
    top_n = predictions[:n]

    result = pd.DataFrame(top_n, columns=['movieId', 'predicted_rating'])
    result = result.merge(movies_df[['movieId', 'title', 'genres']], on='movieId')
    return result

print(recommend(user_id=1, n=10))

   movieId  predicted_rating                             title  \
0     2859          4.517663          Stop Making Sense (1984)   
1     2360          4.463899  Celebration, The (Festen) (1998)   
2     3801          4.454956        Anatomy of a Murder (1959)   
3      318          4.440568  Shawshank Redemption, The (1994)   
4    65133          4.423727    Blackadder Back & Forth (1999)   
5     8477          4.414032                  Jetée, La (1962)   
6     2203          4.406034          Shadow of a Doubt (1943)   
7     3728          4.396764             One False Move (1992)   
8   109687          4.393706             Particle Fever (2013)   
9      903          4.386957                    Vertigo (1958)   

                           genres  
0             Documentary|Musical  
1                           Drama  
2                   Drama|Mystery  
3                     Crime|Drama  
4                          Comedy  
5                  Romance|Sci-Fi  
6            Crime|Dr

In [15]:
from sklearn.preprocessing import LabelEncoder
movies_encoder = LabelEncoder()
users_encoder = LabelEncoder()

merged_copy = merged_df.copy()

merged_copy['movie_idx'] = movies_encoder.fit_transform(merged_copy['movieId'])
merged_copy['user_idx'] = users_encoder.fit_transform(merged_copy['userId'])

n_users  = merged_copy['user_idx'].max() + 1
n_movies = merged_copy['movie_idx'].max() + 1

In [16]:
import scipy.sparse as sparse
user_item = sparse.csr_matrix((
    merged_copy['rating'],
    (
        merged_copy['user_idx'].values,
        merged_copy['movie_idx'].values
    )
    ),shape = (n_users , n_movies)
    ) 

In [17]:
import implicit

model = implicit.als.AlternatingLeastSquares(
    factors=64,         
    regularization=0.1, 
    iterations=20,       
    calculate_training_loss=True,
    random_state=42
)

model.fit(user_item)
print("ALS training complete.")

/Users/ahmedsherif/house-prices-ml/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 20/20 [01:49<00:00,  5.47s/it, loss=0.0148]

ALS training complete.


In [18]:
def ALS_recommend(real_id):
    user_idx = users_encoder.transform([real_id])[0]

    movie_indices , scores = model.recommend(
        userid = user_idx,
        user_items = user_item[user_idx],
        N = 10
    )
    real_movie_ids = movies_encoder.inverse_transform(movie_indices)

    recommended_df = pd.DataFrame({
         'movieId': real_movie_ids,
         'score': scores
})

    final_recommendations = pd.merge(recommended_df, movies_df, on='movieId')
    return final_recommendations

ALS_recommend(3)

,movieId,score,title,genres
0,4370,1.177602,A.I. Artificial Intelligence (2001),Adventure|Drama|Sci-Fi
1,8972,1.152896,National Treasure (2004),Action|Adventure|Drama|Mystery|Thriller
2,608,1.090408,Fargo (1996),Comedy|Crime|Drama|Thriller
3,115149,1.075915,John Wick (2014),Action|Thriller
4,53322,1.068200,Ocean's Thirteen (2007),Crime|Thriller
5,30812,1.066138,"Aviator, The (2004)",Drama
6,5010,1.021524,Black Hawk Down (2001),Action|Drama|War
7,4270,1.010739,"Mummy Returns, The (2001)",Action|Adventure|Comedy|Thriller
8,119145,1.007584,Kingsman: The Secret Service (2015),Action|Adventure|Comedy|Crime
9,122904,0.995282,Deadpool (2016),Action|Adventure|Comedy|Sci-Fi


In [ ]:
tmdb_movies= pd.read_csv('/Users/ahmedsherif/Downloads/archive (5)/tmdb_5000_movies.csv')
tmdb_movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [21]:
tmdb_movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   str    
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

In [22]:
print(tmdb_movies[['title', 'genres','keywords']].head(2))

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   

                                              genres  \
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   

                                            keywords  
0  [{"id": 1463, "name": "culture clash"}, {"id":...  
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...  


In [23]:
tmdb_credits = pd.read_csv('/Users/ahmedsherif/Downloads/archive (5)/tmdb_5000_credits.csv')

In [24]:
full_dataset = pd.merge(tmdb_movies, tmdb_credits, left_on='id', right_on='movie_id')

full_dataset = full_dataset.drop(columns=['title_y', 'movie_id'])
full_dataset = full_dataset.rename(columns={'title_x': 'title'})

print(full_dataset[['id', 'title', 'genres', 'keywords', 'cast', 'crew']].head())

       id                                     title  \
0   19995                                    Avatar   
1     285  Pirates of the Caribbean: At World's End   
2  206647                                   Spectre   
3   49026                     The Dark Knight Rises   
4   49529                               John Carter   

                                              genres  \
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                            keywords  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...   
2  [{"id": 470, "name": "spy"}, {"id": 818, "name...   
3  [{"id": 849, "name": "dc comics"}, {"id": 853,...   
4  [{"id": 818, "name": "based on novel"}, {"id":...

In [25]:
tmdb_df = full_dataset[['title','genres','keywords','cast','crew']]
tmdb_df

,title,genres,keywords,cast,crew
0,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,The Dark Knight Rises,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,John Carter,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."
...,...,...,...,...,...
4798,El Mariachi,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 5616, ""name"": ""united states\u2013mexi...","[{""cast_id"": 1, ""character"": ""El Mariachi"", ""c...","[{""credit_id"": ""52fe44eec3a36847f80b280b"", ""de..."
4799,Newlyweds,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",[],"[{""cast_id"": 1, ""character"": ""Buzzy"", ""credit_...","[{""credit_id"": ""52fe487dc3a368484e0fb013"", ""de..."
4800,"Signed, Sealed, Delivered","[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...","[{""id"": 248, ""name"": ""date""}, {""id"": 699, ""nam...","[{""cast_id"": 8, ""character"": ""Oliver O\u2019To...","[{""credit_id"": ""52fe4df3c3a36847f8275ecf"", ""de..."
4801,Shanghai Calling,[],[],"[{""cast_id"": 3, ""character"": ""Sam"", ""credit_id...","[{""credit_id"": ""52fe4ad9c3a368484e16a36b"", ""de..."


In [ ]:
import ast

def extract_names(val, key='name', top_n=3):
    try:
        items = ast.literal_eval(val)
        
        names = [item[key] for item in items[:top_n]]
        
        return " ".join(names)
    except:
        return ""

tmdb_df['genre_parsed'] = tmdb_df['genres'].fillna('[]').apply(extract_names)
tmdb_df['keywords_parsed'] = tmdb_df['keywords'].fillna('[]').apply(extract_names)
tmdb_df['cast_parsed'] = tmdb_df['cast'].fillna('[]').apply(extract_names)

In [34]:
tmdb_df['cast_parsed']

0         Sam Worthington Zoe Saldana Sigourney Weaver
1            Johnny Depp Orlando Bloom Keira Knightley
2             Daniel Craig Christoph Waltz Léa Seydoux
3             Christian Bale Michael Caine Gary Oldman
4           Taylor Kitsch Lynn Collins Samantha Morton
                             ...                      
4798    Carlos Gallardo Jaime de Hoyos Peter Marquardt
4799          Edward Burns Kerry Bishé Marsha Dietlein
4800            Eric Mabius Kristin Booth Crystal Lowe
4801             Daniel Henney Eliza Coupe Bill Paxton
4802     Drew Barrymore Brian Herzlinger Corey Feldman
Name: cast_parsed, Length: 4803, dtype: str

In [36]:
tmdb_df['crew'].iloc[1]

'[{"credit_id": "52fe4232c3a36847f800b579", "department": "Camera", "gender": 2, "id": 120, "job": "Director of Photography", "name": "Dariusz Wolski"}, {"credit_id": "52fe4232c3a36847f800b4fd", "department": "Directing", "gender": 2, "id": 1704, "job": "Director", "name": "Gore Verbinski"}, {"credit_id": "52fe4232c3a36847f800b54f", "department": "Production", "gender": 2, "id": 770, "job": "Producer", "name": "Jerry Bruckheimer"}, {"credit_id": "52fe4232c3a36847f800b503", "department": "Writing", "gender": 2, "id": 1705, "job": "Screenplay", "name": "Ted Elliott"}, {"credit_id": "52fe4232c3a36847f800b509", "department": "Writing", "gender": 2, "id": 1706, "job": "Screenplay", "name": "Terry Rossio"}, {"credit_id": "52fe4232c3a36847f800b57f", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "52fe4232c3a36847f800b585", "department": "Editing", "gender": 2, "id": 1722, "job": "Editor", "name": "Craig Wood"}, {"credit_id": "52f

In [38]:
import ast

def extract_director(val, key='name'):
    try:
        items = ast.literal_eval(val)
        
        name = [item[key] for item in items if item.get('job') == 'Director']
        
        return " ".join(name)
    except:
        return ''

tmdb_df['crew_parsed'] = tmdb_df['crew'].fillna('[]').apply(extract_director)


In [41]:
tmdb_df = tmdb_df.drop(['genres','keywords','cast','crew'],axis = 1)

In [43]:
tmdb_df = tmdb_df.drop('director_parsed',axis = 1)

In [44]:
tmdb_df

,title,genre_parsed,keywords_parsed,cast_parsed,crew_parsed
0,Avatar,Action Adventure Fantasy,culture clash future space war,Sam Worthington Zoe Saldana Sigourney Weaver,James Cameron
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,ocean drug abuse exotic island,Johnny Depp Orlando Bloom Keira Knightley,Gore Verbinski
2,Spectre,Action Adventure Crime,spy based on novel secret agent,Daniel Craig Christoph Waltz Léa Seydoux,Sam Mendes
3,The Dark Knight Rises,Action Crime Drama,dc comics crime fighter terrorist,Christian Bale Michael Caine Gary Oldman,Christopher Nolan
4,John Carter,Action Adventure Science Fiction,based on novel mars medallion,Taylor Kitsch Lynn Collins Samantha Morton,Andrew Stanton
...,...,...,...,...,...
4798,El Mariachi,Action Crime Thriller,united states–mexico barrier legs arms,Carlos Gallardo Jaime de Hoyos Peter Marquardt,Robert Rodriguez
4799,Newlyweds,Comedy Romance,,Edward Burns Kerry Bishé Marsha Dietlein,Edward Burns
4800,"Signed, Sealed, Delivered",Comedy Drama Romance,date love at first sight narration,Eric Mabius Kristin Booth Crystal Lowe,Scott Smith
4801,Shanghai Calling,,,Daniel Henney Eliza Coupe Bill Paxton,Daniel Hsia


In [46]:
tmdb_df['profile'] = (
    tmdb_df['genre_parsed']   + ' ' +
    tmdb_df['keywords_parsed'] + ' ' +
    tmdb_df['cast_parsed']     + ' ' +
    tmdb_df['crew_parsed'] + ' ' +
    tmdb_df['crew_parsed']    # doubled
)

# Look at a real profile
dark_knight = tmdb_df[tmdb_df['title'] == 'The Dark Knight']['profile'].values[0]
print("Dark Knight profile:")
print(dark_knight)

Dark Knight profile:
Drama Action Crime dc comics crime fighter secret identity Christian Bale Heath Ledger Aaron Eckhart Christopher Nolan Christopher Nolan


In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',   
    max_features=10000      
)


tfidf_matrix = tfidf.fit_transform(tmdb_df['profile'].fillna(''))

print(tfidf_matrix.shape)

(4803, 10000)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from difflib import get_close_matches

movies_df['title_clean'] = (movies_df['title']
    .str.replace(r'\s*\(\d{4}\)', '', regex=True)
    .str.strip()
)
tmdb_df['title_clean'] = tmdb_df['title'].str.strip()

title_to_idx = pd.Series(
    tmdb_df.index, 
    index=tmdb_df['title_clean']
).drop_duplicates()


def recommend_content(movie_title, n=10):
    
    if movie_title not in title_to_idx:
        close = get_close_matches(movie_title, title_to_idx.index, n=1, cutoff=0.6)
        if not close:
            return f"'{movie_title}' not found."
        movie_title = close[0]
        print(f"Did you mean: '{movie_title}'?")
    
    idx = title_to_idx[movie_title]
    
    movie_vec = tfidf_matrix[idx]   

    sim_scores = cosine_similarity(movie_vec, tfidf_matrix).flatten()
    

    top_indices = np.argsort(sim_scores)[::-1][1:n+1]
    
    result = tmdb_df.iloc[top_indices][['title', 'genre_parsed']].copy()
    result['similarity'] = sim_scores[top_indices].round(3)
    result = result.reset_index(drop=True)
    
    return result

print(recommend_content("Inception", n=10))
print("---")
print(recommend_content("Toy Story", n=10))

                   title                     genre_parsed  similarity
0           The Prestige           Drama Mystery Thriller       0.450
1           Interstellar  Adventure Drama Science Fiction       0.432
2               Insomnia           Crime Mystery Thriller       0.425
3                Memento                 Mystery Thriller       0.422
4  The Dark Knight Rises               Action Crime Drama       0.420
5          Batman Begins               Action Crime Drama       0.417
6        The Dark Knight               Drama Action Crime       0.403
7                Don Jon             Romance Comedy Drama       0.328
8               Fortress  Action Thriller Science Fiction       0.199
9    Highlander: Endgame   Action Fantasy Science Fiction       0.195
---
                title                genre_parsed  similarity
0         Toy Story 2     Animation Comedy Family       0.622
1              Cars 2  Animation Family Adventure       0.394
2        A Bug's Life  Adventure Animati